# GraphConsis & BWGNN — Colab experiments

Train **GraphConsis** and **BWGNN** on YelpChi, Amazon, T-Finance, and T-Social.

**Easiest Colab setup:** zip the whole `Graph-Fraud-WK` folder (code + `dataset/`), upload that zip to Google Drive, then unpack it in the next cell.

Each experiment cell has a parameter dict you can edit. After training, artifacts are written under:

`notebook_runs/<Model>/<Dataset>/seed<seed>_<timestamp>_ep…/`

(config, metrics, split, predictions, checkpoint), then that folder is **zipped**. On Colab the zip downloads automatically.

**Runtime → Change runtime type → GPU** before installing packages.

T-Finance (~42M edges) and T-Social (multi-GB) are heavy. Prefer a GPU High-RAM runtime for those cells.

## 1. Unpack the repo zip from Google Drive

Zip **the project folder** on your computer (the one that contains `algorithms/` and `dataset/`), upload it to Drive, then set `DRIVE_REPO_ZIP` below.

Include in the zip:
- the code (`algorithms/`, `utils/`, this notebook, …)
- `dataset/` with either the extracted files (`YelpChi.mat`, `Amazon.mat`, `tfinance`, `tsocial`) **or** the inner zips (`YelpChi.zip`, …) — those inner zips are unpacked automatically

Exclude `.venv` (a Windows venv will not work on Colab and makes the zip huge). Exclude `results/` / `notebook_runs/` if you want a smaller upload.

Example: file at `My Drive / Graph-Fraud-WK.zip` → `DRIVE_REPO_ZIP = "Graph-Fraud-WK.zip"`

In [4]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Zip of the whole repo (code + dataset/) on Google Drive, relative to My Drive.
DRIVE_REPO_ZIP = "/content/drive/MyDrive/sharare/colab_graphconsis_bwgnn.zip"
UNPACK_DEST = "/content" if IN_COLAB else str(Path.cwd() / "_unpacked_repo")

In [5]:
def _looks_like_repo(path):
    return (Path(path) / "algorithms" / "GraphConsis" / "supervised_train.py").is_file()


def _find_repo(starts):
    seen = set()
    for start in starts:
        start = Path(start)
        if not start.exists():
            continue
        queue = [start]
        if start.is_dir():
            queue.extend(sorted(p for p in start.iterdir() if p.is_dir()))
        for candidate in queue:
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if _looks_like_repo(candidate):
                return candidate.resolve()
    dest = Path(UNPACK_DEST)
    if dest.is_dir():
        for hit in dest.rglob("supervised_train.py"):
            if hit.parent.name == "GraphConsis":
                return hit.parents[2].resolve()
    return None


def _resolve_zip(name):
    raw = str(name).strip().replace("\\", "/")
    p = Path(raw).expanduser()
    if p.is_file():
        return p.resolve()
    mydrive = Path("/content/drive/MyDrive")
    candidates = [mydrive / raw, Path("/content/drive") / raw, Path.cwd() / raw]
    if raw.lower().startswith("mydrive/"):
        candidates.insert(0, Path("/content/drive") / raw)
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Repo zip not found: {!r}. After mounting Drive, set DRIVE_REPO_ZIP "
        "relative to My Drive (e.g. 'Graph-Fraud-WK.zip').".format(name)
    )


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
zip_path = _resolve_zip(DRIVE_REPO_ZIP)
dest = Path(UNPACK_DEST)

print("Unpacking", zip_path, "->", dest)
unzip_bin = shutil.which("unzip")
if unzip_bin:
    subprocess.check_call([unzip_bin, "-o", "-q", str(zip_path), "-d", str(dest)])
else:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)

repo = dest

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Repo root:", Path.cwd().resolve())
print("Colab:", IN_COLAB)

Unpacking /content/drive/MyDrive/sharare/colab_graphconsis_bwgnn.zip -> /content
Repo root: /content
Colab: True


## 2. Install packages (Colab)

Colab already ships CUDA PyTorch. Install PyG and sympy only — do **not** `pip install dgl` (broken on many Colab / Windows Python builds). T-Finance / T-Social load without DGL.

Re-run this cell after a runtime restart.

In [7]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

%pip install -q torch-geometric sympy

import torch_geometric
import sympy
print("torch-geometric", torch_geometric.__version__, "sympy", sympy.__version__)

torch 2.11.0+cu128 cuda True NVIDIA A100-SXM4-40GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.0 MB/s eta 0:00:00
torch-geometric 2.8.0.post1 sympy 1.14.0


## 3. Check datasets

If `dataset/` inside the repo zip already has `YelpChi.mat`, `Amazon.mat`, `tfinance`, and `tsocial`, this cell only verifies them.

If you zipped the **inner** archives (`YelpChi.zip`, `Amazon.zip`, `tfinance.zip`, `tsocial.zip`) instead, they are unpacked here.

Leave `DRIVE_DATASET_DIR` empty unless the graphs live in a **separate** Drive folder (not inside the repo zip).

In [8]:
from utils.notebook_runner import (
    dataset_dir,
    missing_dataset_files,
    prepare_datasets_from_drive,
    unzip_if_needed,
)

# Only if the graphs are NOT already under the unzipped repo's dataset/.
# Relative to My Drive, e.g. "GraphFraud/zips"
DRIVE_DATASET_DIR = ""

ddir = dataset_dir()
unzip_if_needed(ddir)

if str(DRIVE_DATASET_DIR).strip():
    missing = prepare_datasets_from_drive(DRIVE_DATASET_DIR, dest=ddir)
else:
    missing = missing_dataset_files(ddir)

print("dataset/:", ddir.resolve())
for p in sorted(ddir.iterdir()):
    if p.is_file() and not p.name.lower().endswith(".zip"):
        print("  {:<24} {:8.1f} MB".format(p.name, p.stat().st_size / 1e6))
if missing:
    print("Still missing:", missing)
    print("Put those files (or their .zip) in dataset/ inside the repo zip, or set DRIVE_DATASET_DIR.")
else:
    print("All four dataset files are present.")

Already present: Amazon.mat
No known dataset files inside /content/dataset/DBLP4057_GAT_with_idx_tra200_val_800.zip
Already present: YelpChi.mat
No known dataset files inside /content/dataset/archive.zip
Already present: tfinance
Already present: tsocial
dataset/: /content/dataset
  Amazon.mat                  222.6 MB
  DBLP4057_GAT_with_idx_tra200_val_800.mat    406.0 MB
  YelpChi.mat                 207.7 MB
  tfinance                    682.9 MB
  tsocial                    4110.3 MB
All four dataset files are present.


## 4. Run helpers

Edit the dict in a cell below, then run that cell. Each run gets its own folder plus a zip.

Set `download=False` if you do not want the Colab download dialog every time (zips are still written next to the folder).

In [9]:
import torch
from utils.notebook_runner import (
    notebook_runs_dir,
    run_bwgnn,
    run_graphconsis,
    zip_all_notebook_runs,
)

print("Outputs:", notebook_runs_dir().resolve())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Outputs: /content/notebook_runs
GPU: NVIDIA A100-SXM4-40GB


---
## GraphConsis

Tunable keys: `epochs`, `learning_rate`, `dim_1`, `batch_size`, `dropout`, `weight_decay`, `context_dim`, `sigmoid`, `samples_1`, `samples_2`, `test_size`, `seed`, `gpu`.

Folder: `notebook_runs/GraphConsis/<Dataset>/seed<seed>_<tag>/`

In [ ]:
# GraphConsis · YelpChi — edit params, then run this cell
GC_YELPCHI = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.2,
    seed=123,
    gpu="0",
)

gc_yelpchi = run_graphconsis("yelpchi", download=True, **GC_YELPCHI)
gc_yelpchi["metrics"]

GraphConsis YelpChi.mat -> /content/notebook_runs
Epoch 0 loss: 0.3661
Epoch 1 loss: 0.2497
Epoch 2 loss: 0.2216
Epoch 3 loss: 0.2057
Epoch 4 loss: 0.1972
Epoch 5 loss: 0.1861
Epoch 6 loss: 0.1805
Epoch 7 loss: 0.1736
Epoch 8 loss: 0.1616
Epoch 9 loss: 0.1544
Epoch 10 loss: 0.1485
Epoch 11 loss: 0.1386
Epoch 12 loss: 0.1331
Epoch 13 loss: 0.1317
Epoch 14 loss: 0.1251
Epoch 15 loss: 0.1307
Epoch 16 loss: 0.1214
Epoch 17 loss: 0.1115
Epoch 18 loss: 0.1074
Epoch 19 loss: 0.0982
Epoch 20 loss: 0.0931
Epoch 21 loss: 0.0891
Epoch 22 loss: 0.0817
Epoch 23 loss: 0.0829
Epoch 24 loss: 0.0763
Epoch 25 loss: 0.0772
Epoch 26 loss: 0.0683
Epoch 27 loss: 0.0922
Epoch 28 loss: 0.0869
Epoch 29 loss: 0.0997
Epoch 30 loss: 0.0773
Epoch 31 loss: 0.0685
Epoch 32 loss: 0.0646
Epoch 33 loss: 0.0624
Epoch 34 loss: 0.0601
Epoch 35 loss: 0.0547
Epoch 36 loss: 0.0524
Epoch 37 loss: 0.0571
Epoch 38 loss: 0.0521
Epoch 39 loss: 0.0463
Epoch 40 loss: 0.0522
Epoch 41 loss: 0.0484
Epoch 42 loss: 0.0361
Epoch 43 loss:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.9108910891089109,
 'f1_micro': 0.9108910891089109,
 'f1_macro': 0.8180115504220347,
 'f1_weighted': 0.9102545403363081,
 'precision_macro': 0.8226616883938742,
 'recall_macro': 0.8135713917938625,
 'g_mean': 0.8019249664400108,
 'auroc': 0.9127841884253646,
 'auprc': 0.7340983095767076,
 'auc_roc': 0.9127841884253646}

In [ ]:
# GraphConsis · Amazon — edit params, then run this cell
GC_AMAZON = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_amazon = run_graphconsis("amazon", download=True, **GC_AMAZON)
gc_amazon["metrics"]

GraphConsis Amazon.mat -> /content/notebook_runs
Epoch 0 loss: 0.4295
Epoch 1 loss: 0.2444
Epoch 2 loss: 0.1749
Epoch 3 loss: 0.1224
Epoch 4 loss: 0.0815
Epoch 5 loss: 0.0745
Epoch 6 loss: 0.0666
Epoch 7 loss: 0.0619
Epoch 8 loss: 0.0576
Epoch 9 loss: 0.0548
Epoch 10 loss: 0.0513
Epoch 11 loss: 0.0469
Epoch 12 loss: 0.0457
Epoch 13 loss: 0.0436
Epoch 14 loss: 0.0429
Epoch 15 loss: 0.0398
Epoch 16 loss: 0.0394
Epoch 17 loss: 0.0377
Epoch 18 loss: 0.0328
Epoch 19 loss: 0.0292
Epoch 20 loss: 0.0370
Epoch 21 loss: 0.0345
Epoch 22 loss: 0.0293
Epoch 23 loss: 0.0290
Epoch 24 loss: 0.0307
Epoch 25 loss: 0.0290
Epoch 26 loss: 0.0318
Epoch 27 loss: 0.0290
Epoch 28 loss: 0.0222
Epoch 29 loss: 0.0204
Epoch 30 loss: 0.0184
Epoch 31 loss: 0.0178
Epoch 32 loss: 0.0163
Epoch 33 loss: 0.0166
Epoch 34 loss: 0.0173
Epoch 35 loss: 0.0199
Epoch 36 loss: 0.0270
Epoch 37 loss: 0.0254
Epoch 38 loss: 0.0185
Epoch 39 loss: 0.0151
Epoch 40 loss: 0.0129
Epoch 41 loss: 0.0114
Epoch 42 loss: 0.0123
Epoch 43 loss: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.9746755964838845,
 'f1_micro': 0.9746755964838845,
 'f1_macro': 0.8942184416679855,
 'f1_weighted': 0.9738067287212938,
 'precision_macro': 0.9236297102787943,
 'recall_macro': 0.8692059468347493,
 'g_mean': 0.8605653721014929,
 'auroc': 0.9717607563716087,
 'auprc': 0.8502561815738011,
 'auc_roc': 0.9717607563716087}

In [11]:
import gc
# 1. Delete your heavy variables (replace with your actual variables)
# del gc_yelpchi
# del gc_amazon
del gc_tfinance

# 2. Force garbage collection
gc.collect()

# 3. Clear the PyTorch CUDA cache
torch.cuda.empty_cache()

In [10]:
# GraphConsis · T-Finance — 42M edges; GPU recommended
GC_TFINANCE = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tfinance = run_graphconsis("tfinance", download=True, **GC_TFINANCE)
gc_tfinance["metrics"]

GraphConsis tfinance -> /content/notebook_runs
Epoch 0 loss: 0.2359
Epoch 1 loss: 0.0930
Epoch 2 loss: 0.0787
Epoch 3 loss: 0.0754
Epoch 4 loss: 0.0716
Epoch 5 loss: 0.0715
Epoch 6 loss: 0.0674
Epoch 7 loss: 0.0645
Epoch 8 loss: 0.0630
Epoch 9 loss: 0.0613
Epoch 10 loss: 0.0659
Epoch 11 loss: 0.0640
Epoch 12 loss: 0.0597
Epoch 13 loss: 0.0581
Epoch 14 loss: 0.0571
Epoch 15 loss: 0.0573
Epoch 16 loss: 0.0571
Epoch 17 loss: 0.0530
Epoch 18 loss: 0.0552
Epoch 19 loss: 0.0516
Epoch 20 loss: 0.0525
Epoch 21 loss: 0.0522
Epoch 22 loss: 0.0519
Epoch 23 loss: 0.0525
Epoch 24 loss: 0.0541
Epoch 25 loss: 0.0504
Epoch 26 loss: 0.0497
Epoch 27 loss: 0.0510
Epoch 28 loss: 0.0504
Epoch 29 loss: 0.0476
Epoch 30 loss: 0.0456
Epoch 31 loss: 0.0486
Epoch 32 loss: 0.0479
Epoch 33 loss: 0.0507
Epoch 34 loss: 0.0484
Epoch 35 loss: 0.0492
Epoch 36 loss: 0.0480
Epoch 37 loss: 0.0465
Epoch 38 loss: 0.0471
Epoch 39 loss: 0.0442
Epoch 40 loss: 0.0469
Epoch 41 loss: 0.0443
Epoch 42 loss: 0.0468
Epoch 43 loss: 0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.9834212030743823,
 'f1_micro': 0.9834212030743823,
 'f1_macro': 0.9040798796832406,
 'f1_weighted': 0.9833159170606631,
 'precision_macro': 0.9092860007263703,
 'recall_macro': 0.8990193968517943,
 'g_mean': 0.8942039926738913,
 'auroc': 0.9649689880151167,
 'auprc': 0.8701807625766096,
 'auc_roc': 0.9649689880151167}

In [12]:
# GraphConsis · T-Social — very large; smaller batch / fewer epochs by default
GC_TSOCIAL = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=64,
    batch_size=256,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=10,
    samples_2=5,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tsocial = run_graphconsis("tsocial", download=True, **GC_TSOCIAL)
gc_tsocial["metrics"]

GraphConsis tsocial -> /content/notebook_runs


ValueError: No 2-D float feature matrix in /content/dataset/tsocial

---
## BWGNN

Tunable keys: `epochs`, `learning_rate`, `hid_dim`, `order`, `weight_decay`, `train_ratio` (overrides `test_size`), `test_size`, `seed`, `gpu`.

Paper defaults: `train_ratio=0.4`; T-Social uses `hid_dim=10`, `order=5`.

Folder: `notebook_runs/BWGNN/<dataset>/seed<seed>_<tag>/`

In [ ]:
# BWGNN · YelpChi — edit params, then run this cell
BW_YELPCHI = dict(
    epochs=50,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=None,
    test_size=0.2,
    seed=123,
    gpu="0",
)

bw_yelpchi = run_bwgnn("yelpchi", download=True, **BW_YELPCHI)
bw_yelpchi["metrics"]

BWGNN yelpchi -> /content/notebook_runs
cross entropy weight: [1.0, 5.8818793296813965]


/content/utils/data_loader.py:171: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  self._adj = torch.sparse_coo_tensor(


Epoch 0 loss: 0.6890
Epoch 1 loss: 0.6154
Epoch 2 loss: 0.7014
Epoch 3 loss: 0.6024
Epoch 4 loss: 0.6198
Epoch 5 loss: 0.6354
Epoch 6 loss: 0.6360
Epoch 7 loss: 0.6265
Epoch 8 loss: 0.6105
Epoch 9 loss: 0.5937
Epoch 10 loss: 0.5870
Epoch 11 loss: 0.5898
Epoch 12 loss: 0.5815
Epoch 13 loss: 0.5631
Epoch 14 loss: 0.5522
Epoch 15 loss: 0.5461
Epoch 16 loss: 0.5373
Epoch 17 loss: 0.5250
Epoch 18 loss: 0.5209
Epoch 19 loss: 0.5345
Epoch 20 loss: 0.5259
Epoch 21 loss: 0.5073
Epoch 22 loss: 0.5294
Epoch 23 loss: 0.4950
Epoch 24 loss: 0.5109
Epoch 25 loss: 0.4906
Epoch 26 loss: 0.5086
Epoch 27 loss: 0.4890
Epoch 28 loss: 0.5005
Epoch 29 loss: 0.4840
Epoch 30 loss: 0.4958
Epoch 31 loss: 0.4833
Epoch 32 loss: 0.4900
Epoch 33 loss: 0.4818
Epoch 34 loss: 0.4826
Epoch 35 loss: 0.4791
Epoch 36 loss: 0.4786
Epoch 37 loss: 0.4783
Epoch 38 loss: 0.4748
Epoch 39 loss: 0.4751
Epoch 40 loss: 0.4717
Epoch 41 loss: 0.4722
Epoch 42 loss: 0.4699
Epoch 43 loss: 0.4694
Epoch 44 loss: 0.4677
Epoch 45 loss: 0.466

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.7464911326297465,
 'f1_micro': 0.7464911326297465,
 'f1_macro': 0.6583157984086777,
 'f1_weighted': 0.7814666560774696,
 'precision_macro': 0.6520451432127562,
 'recall_macro': 0.7774038974957473,
 'g_mean': 0.7761819921504448,
 'auroc': 0.8576576409071145,
 'auprc': 0.5315387903781617,
 'auc_roc': 0.8576576409071145}

In [ ]:
# BWGNN · Amazon — edit params, then run this cell
BW_AMAZON = dict(
    epochs=50,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_amazon = run_bwgnn("amazon", download=True, **BW_AMAZON)
bw_amazon["metrics"]

BWGNN amazon -> /content/notebook_runs
cross entropy weight: [1.0, 13.564023971557617]
Epoch 0 loss: 0.9154
Epoch 1 loss: 24.5240
Epoch 2 loss: 5.3717
Epoch 3 loss: 3.2552
Epoch 4 loss: 1.7826
Epoch 5 loss: 0.7116
Epoch 6 loss: 0.6582
Epoch 7 loss: 0.6735
Epoch 8 loss: 0.4757
Epoch 9 loss: 0.3488
Epoch 10 loss: 0.3033
Epoch 11 loss: 0.2863
Epoch 12 loss: 0.2746
Epoch 13 loss: 0.2612
Epoch 14 loss: 0.4279
Epoch 15 loss: 0.2579
Epoch 16 loss: 0.2602
Epoch 17 loss: 0.2604
Epoch 18 loss: 0.2562
Epoch 19 loss: 0.2512
Epoch 20 loss: 0.2487
Epoch 21 loss: 0.2476
Epoch 22 loss: 0.2450
Epoch 23 loss: 0.2391
Epoch 24 loss: 0.2335
Epoch 25 loss: 0.2296
Epoch 26 loss: 0.2263
Epoch 27 loss: 0.2212
Epoch 28 loss: 0.2243
Epoch 29 loss: 0.2136
Epoch 30 loss: 0.2134
Epoch 31 loss: 0.2057
Epoch 32 loss: 0.2011
Epoch 33 loss: 0.1951
Epoch 34 loss: 0.1925
Epoch 35 loss: 0.1874
Epoch 36 loss: 0.1861
Epoch 37 loss: 0.1832
Epoch 38 loss: 0.1823
Epoch 39 loss: 0.1771
Epoch 40 loss: 0.1731
Epoch 41 loss: 0.171

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.9681875261615739,
 'f1_micro': 0.9681875261615739,
 'f1_macro': 0.8828468189443799,
 'f1_weighted': 0.9690804156927003,
 'precision_macro': 0.8627380308927499,
 'recall_macro': 0.9058977315622186,
 'g_mean': 0.9030138742123714,
 'auroc': 0.973883393581462,
 'auprc': 0.860774551108074,
 'auc_roc': 0.973883393581462}

In [ ]:
# BWGNN · T-Finance — paper: hid 64, order 2, 40% train
BW_TFINANCE = dict(
    epochs=50,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tfinance = run_bwgnn("tfinance", download=True, **BW_TFINANCE)
bw_tfinance["metrics"]

BWGNN tfinance -> /content/notebook_runs
cross entropy weight: [1.0, 20.80332374572754]
Epoch 0 loss: 63.2826
Epoch 1 loss: 114.5408
Epoch 2 loss: 46.7390
Epoch 3 loss: 9.5479
Epoch 4 loss: 21.8405
Epoch 5 loss: 13.3438
Epoch 6 loss: 2.0928
Epoch 7 loss: 4.2619
Epoch 8 loss: 3.7174
Epoch 9 loss: 2.8011
Epoch 10 loss: 1.4286
Epoch 11 loss: 0.8381
Epoch 12 loss: 1.1823
Epoch 13 loss: 1.0476
Epoch 14 loss: 0.7226
Epoch 15 loss: 0.6684
Epoch 16 loss: 0.5878
Epoch 17 loss: 0.4443
Epoch 18 loss: 0.4061
Epoch 19 loss: 0.4472
Epoch 20 loss: 0.4779
Epoch 21 loss: 0.4563
Epoch 22 loss: 0.4022
Epoch 23 loss: 0.3557
Epoch 24 loss: 0.3401
Epoch 25 loss: 0.3452
Epoch 26 loss: 0.3487
Epoch 27 loss: 0.3404
Epoch 28 loss: 0.3272
Epoch 29 loss: 0.3169
Epoch 30 loss: 0.3104
Epoch 31 loss: 0.3040
Epoch 32 loss: 0.2969
Epoch 33 loss: 0.2879
Epoch 34 loss: 0.2782
Epoch 35 loss: 0.2709
Epoch 36 loss: 0.2698
Epoch 37 loss: 0.2717
Epoch 38 loss: 0.2672
Epoch 39 loss: 0.2588
Epoch 40 loss: 0.2548
Epoch 41 loss:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'accuracy': 0.9565530383230997,
 'f1_micro': 0.9565530383230997,
 'f1_macro': 0.8128572736782415,
 'f1_weighted': 0.9618167211917414,
 'precision_macro': 0.7545317725752508,
 'recall_macro': 0.9182844418041052,
 'g_mean': 0.9173175288720529,
 'auroc': 0.9550473640919176,
 'auprc': 0.8416630529778208,
 'auc_roc': 0.9550473640919176}

In [ ]:
# BWGNN · T-Social — paper: hid 10, order 5; needs a large GPU
BW_TSOCIAL = dict(
    epochs=50,
    learning_rate=0.01,
    hid_dim=10,
    order=5,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tsocial = run_bwgnn("tsocial", download=True, **BW_TSOCIAL)
bw_tsocial["metrics"]

BWGNN tsocial -> /content/notebook_runs


ValueError: No 2-D float feature matrix in /content/dataset/tsocial

## Download every notebook run

Optional: one zip of `notebook_runs/` after you have finished several cells.

In [ ]:
zip_all_notebook_runs(download=True)